# DR03 — Text Conditioning

## 1. Objective and scope

DR01 froze data, compositional splits, captions, and the tokenizer. DR02 froze the pixel-space DDPM and Tiny U-Net mechanics. DR03 adds a from-scratch text encoder, bottleneck image-to-text conditioning, conditional training, and validation-only checkpoint selection. **ID_TEST was not used. FINAL_OOD was not used.**

In [ ]:
from __future__ import annotations

import csv
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = next(parent for parent in Path.cwd().resolve().parents if (parent / 'src').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
CANONICAL = ROOT / 'reports' / 'dr03_canonical'
summary = json.loads((CANONICAL / 'dr03_summary.json').read_text(encoding='utf-8'))
print(f'Project root: {ROOT.name}; canonical run: {summary["run_id"]}')

## 2. Frozen DR01 / DR02 inputs

DR03 uses Google Cartoon Set 10k at 32×32 with TRAIN = 8,388. The DR01 caption grammar is `avatar <face_shape> <glasses> <hair_color> <face_color>`, tokenized from TRAIN only with a 45-token vocabulary and maximum length 7. DR02 supplies pixel-space DDPM epsilon prediction with epsilon-MSE, T=1000, a linear beta schedule, fixed-small variance, and an 8×8 bottleneck `[B,128,8,8]`.

This notebook summarizes the DR03 handoff and does not duplicate prior notebooks.

## 3. Canonical text representation

This frozen **ID_VAL** example is embedded from the fixed preview metadata; it is not a test or FINAL_OOD example. The caption, serialized tokens, IDs, and mask were recorded before this notebook. `normalize_and_split` is the production DR01 utility.

In [ ]:
from tiny_avatar_diffusion.tokenizer import normalize_and_split

example = {
    'split': 'ID_VAL',
    'caption': 'avatar face_shape_wide glasses_thick_rectangular_glasses hair_color_silver face_color_deep_brown',
    'tokens': ['<bos>', 'avatar', 'face_shape_wide', 'glasses_thick_rectangular_glasses', 'hair_color_silver', 'face_color_deep_brown', '<eos>'],
    'token_ids': [1, 4, 22, 34, 44, 6, 2],
    'attention_mask': [1, 1, 1, 1, 1, 1, 1],
}
assert normalize_and_split(example['caption']) == example['tokens'][1:-1]
for key in ('caption', 'tokens', 'token_ids', 'attention_mask'):
    print(f'{key}: {example[key]}')
print('serialized shape:', (1, len(example['token_ids'])))

## 4. From-scratch text encoder

The production encoder has learned token embeddings; d_model=64; two Pre-LN Transformer layers; four 16-dimensional heads; FFN=256; GELU; sinusoidal positions; attention/FFN dropout 0.1; embedding dropout 0; final LayerNorm; PAD masking; no causal mask; no CLS. Its output is `[B,L,64]`.

The small CPU check below verifies the frozen parameter budget (102,976) and output shape; it does not load a checkpoint or train.

In [ ]:
from tiny_avatar_diffusion.text_conditioning import TextConditioningEncoder

def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters())

text_encoder = TextConditioningEncoder().eval()
input_ids = torch.tensor([example['token_ids']], dtype=torch.long)
attention_mask = torch.tensor([example['attention_mask']], dtype=torch.long)
with torch.no_grad():
    context = text_encoder(input_ids, attention_mask)
assert count_parameters(text_encoder) == 102_976
assert tuple(context.shape) == (1, 7, 64)
print('text encoder parameters:', count_parameters(text_encoder))
print('context shape:', tuple(context.shape))

## 5. Bottleneck cross-attention

One block acts only at the bottleneck. Image features `[B,128,8,8]` supply queries; text context `[B,L,64]` supplies keys and values. Attention dimension is 64 with four heads, query Pre-LN, residual return to 128 channels, PAD masking, no visual self-attention, and dropout 0. The tensor sequence walkthrough is `[B,128,8,8] → [B,64,128] → [B,4,64,16]`, then back to `[B,128,8,8]`.

In [ ]:
from tiny_avatar_diffusion.text_conditioning import ImageTextCrossAttention

cross_attention = ImageTextCrossAttention().eval()
visual = torch.zeros(1, 128, 8, 8)
with torch.no_grad():
    attended = cross_attention(visual, context, attention_mask)
assert count_parameters(cross_attention) == 25_152
assert tuple(attended.shape) == (1, 128, 8, 8)
print('cross-attention parameters:', count_parameters(cross_attention))
print('input/output shapes:', tuple(visual.shape), '->', tuple(attended.shape))

## 6. Conditional parameter budget

The conditional addition is deliberately small and localized.

In [ ]:
from tiny_avatar_diffusion.unet import ConditionalTinyUNet, TinyUNet

unconditional_unet = TinyUNet().eval()
conditional_unet = ConditionalTinyUNet().eval()
budget = {
    'DR02 unconditional Tiny U-Net': count_parameters(unconditional_unet),
    'Cross-attention': count_parameters(cross_attention),
    'Conditional Tiny U-Net': count_parameters(conditional_unet),
    'Text encoder': count_parameters(text_encoder),
    'Full conditional system': count_parameters(conditional_unet) + count_parameters(text_encoder),
}
expected = [4_595_779, 25_152, 4_620_931, 102_976, 4_723_907]
assert list(budget.values()) == expected
for name, value in budget.items():
    print(f'{name:34} {value:,}')

## 7. Frozen training configuration

The compact configuration below is read from the canonical DR03 summary. Image presentations refer only to successful optimizer updates; they are not a claim about total forward passes.

In [ ]:
configuration = {
    'dataset': summary['dataset'], 'resolution': summary['resolution'], 'TRAIN size': summary['train_size'],
    'seed': summary['seed'], 'loss': summary['diffusion']['loss'], 'timesteps': summary['diffusion']['timesteps'],
    'beta schedule': summary['diffusion']['beta_schedule'], 'optimizer': summary['optimizer']['name'],
    'learning rate': summary['optimizer']['learning_rate'], 'weight decay': summary['optimizer']['weight_decay'],
    'physical batch': summary['training']['physical_batch_size'], 'precision': summary['training']['precision'],
    'successful optimizer steps': summary['training']['successful_optimizer_steps'],
    'successful-update image presentations': summary['training']['successful_update_image_presentations'],
    'skipped AMP update attempts': summary['training']['amp_skipped_update_attempts'], 'EMA decay': summary['ema']['decay'],
}
for key, value in configuration.items():
    print(f'{key:38} {value}')

## 8. AMP and EMA semantics

EMA begins as an exact raw copy at step 0, covers both the text encoder and U-Net, and updates only after successful optimizer updates; the successful-step counter follows the same rule. The earlier r1 run stopped at successful step 10,076 because training-loop handling of a GradScaler overflow/non-finite-gradient skipped update was incomplete, not because of a scientific model failure. The authoritative r2 run restarted from step 0 and completed 20,000 successful updates.

## 9. Training / validation trajectory

The canonical CSV contains raw train epsilon-MSE, a trailing 100-observation mean, and RAW-weight validation diagnostics. It shows a strong overall downward trend with intermediate rebounds; COMP_VAL is generally above ID_VAL without progressive divergence. The loss-preferred final milestone is 20k. This is not a claim of compositional generalization.

In [ ]:
with (CANONICAL / 'training_validation_metrics.csv').open(newline='', encoding='utf-8') as handle:
    metrics = {int(row['step']): row for row in csv.DictReader(handle)}
print(' step | ID_VAL RAW epsilon-MSE | COMP_VAL RAW epsilon-MSE')
for step in (5000, 10000, 15000, 20000):
    row = metrics[step]
    print(f'{step:5} | {float(row["id_val_epsilon_mse_raw"]):.15f} | {float(row["comp_val_epsilon_mse_raw"]):.15f}')
figure = plt.imread(CANONICAL / 'training_validation_epsilon_mse.png')
plt.figure(figsize=(9, 5)); plt.imshow(figure); plt.axis('off'); plt.show()

## 10. Fixed EMA qualitative previews

Rows are ID_VAL seeds 2026 and 2027, then COMP_VAL seeds 2028 and 2029; columns are 5k, 10k, 15k, and 20k. The protocol fixes prompts and seeds and uses EMA, T=1000, and `clip_denoised=True`. **Qualitative diagnostic only**: it does not establish quantitative conditioning adherence or final image quality.

In [ ]:
figure = plt.imread(CANONICAL / 'fixed_ema_preview_milestones.png')
plt.figure(figsize=(9, 8)); plt.imshow(figure); plt.axis('off'); plt.show()

## 11. Validation-only KID checkpoint guardrail

The manifest has 512 total entries: ID_VAL=256 and COMP_VAL=256. Therefore validation_samples_per_split=256, not 512. Every row uses EMA and Clean-FID 0.1.35 in clean mode. The frozen rule `COMP_KID_20k <= COMP_KID_15k` passed, selecting 20k EMA. This KID is a **validation-only checkpoint-selection guardrail**, not the final project KID evaluation.

In [ ]:
with (CANONICAL / 'checkpoint_selection_kid.csv').open(newline='', encoding='utf-8') as handle:
    kid_rows = list(csv.DictReader(handle))
assert {row['validation_samples_per_split'] for row in kid_rows} == {'256'}
print(' step | weight | ID_VAL KID          | COMP_VAL KID        | samples/split')
for row in kid_rows:
    print(f"{row['step']:>5} | {row['weight_set']:>6} | {float(row['id_val_kid']):.15f} | {float(row['comp_val_kid']):.15f} | {row['validation_samples_per_split']}")
figure = plt.imread(CANONICAL / 'validation_kid_by_milestone.png')
plt.figure(figsize=(9, 5)); plt.imshow(figure); plt.axis('off'); plt.show()

## 12. Frozen checkpoint and provenance

The selected checkpoint is 20k EMA. The internal manifest identity hashes the canonical payload before its `sha256` field; the physical SHA hashes the actual serialized JSON bytes. They are distinct definitions, not a conflict.

In [ ]:
provenance = summary['provenance']
selection = summary['checkpoint_selection']
print('selected checkpoint:', selection['selected_step'], selection['weight_set'])
print('checkpoint SHA256:', selection['selected_checkpoint_sha256'])
print('training commit:', provenance['training_git_commit'])
print('evaluation commit:', provenance['evaluation_git_commit'])
print('manifest internal identity:', provenance['validation_manifest_sha256'])
print('manifest physical-file SHA256:', provenance['authoritative_source_file_sha256']['reports/dr03_checkpoint_selection_hotfix/validation_quality_manifest.json'])

## 13. DR03 conclusions and boundary

DR03 implemented a from-scratch text encoder and bottleneck cross-attention while retaining a tiny conditional system. The authoritative r2 run completed after AMP-accounting repair; optimization and validation-only KID improved strongly, and the frozen rule selected 20k EMA for later evaluation.

**ID_TEST USED: NO**  
**FINAL_OOD USED: NO**  
**Final image-quality evaluation: NOT YET PERFORMED**  
**Dedicated conditioning-adherence evaluation: NOT YET PERFORMED**

DR03 does not establish final compositional-generalization performance.